# Preprocess Pilot Data 

## Goals:

1. **Data Import**
    - Import EEG data into MNE object.

2. **Preprocessing**
    - Preprocess the EEG data.
        * Bandpass filter
        * Autoreject bad channels
    - Epoch data based on stimulus appearance.
    
3. **Export Data**
    - Save organized epochs to an npz file
    - Save preprocessing settings to a json file

# Import Libraries

In [ ]:
# Import libraries
import mne
from autoreject import Ransac 
import json
import numpy as np
import matplotlib.pyplot as plt

# Import custom scripts
from Functions import import_data
from Functions import data_tools

# Enable interactive plots
%matplotlib qt

# Import Data

Import EEG data
- Set channel names
- Create an mne raw object
- Set MNE channel montage

In [ ]:
# import EEG data and fill settings
#file_name = r"Pilot2\EEG\sub-P007\ses-S001\eeg\sub-P007_ses-S001_task-T1_run-001_eeg.xdf"
file_name = r"PedsData\EEG\sub-P047\ses-S001\eeg\sub-P047_ses-S001_task-T1_run-001_eeg.xdf"
ch_names = ["Fz", "F4", "F8", "C3", "Cz", "C4", "T8", "P7", "P3", "P4", "P8", "PO7", "PO8", "O1", "Oz", "O2"]
#ch_names = ["Fz", "Cz", "P3", "Pz", "P4", "PO7", "Oz", "PO8"] this is the montage for mock EEG data

[eeg_ts, eeg_data, eeg_fs] = import_data.read_xdf(f"Data\\{file_name}", picks=ch_names)

# Create MNE array
info = mne.create_info(ch_names, eeg_fs, ch_types = 'eeg')  # Create info properties
mne_raw = mne.io.RawArray(eeg_data, info = info)            

# Set standard channel montageR
mne_raw.set_montage('standard_1020')  

# Plot raw data
mne_raw.plot(title="RAW data")   

# Bandpass Filter Data

Stimulation frequency is 10 Hz, so filter between 0.5 Hz (highpass) and 35 Hz (lowpass)

In [ ]:
# Settings
hpf_fc = 0.5   # High-pass cut-off frequency: list[Hz]
lpf_fc = 35    # Low-pass cut-off frequency: list[Hz]

# Apply bandpass filter
filt_raw = mne_raw.copy().filter(l_freq=hpf_fc, h_freq=lpf_fc, picks=ch_names)

# Plot filtered data
filt_raw.plot(scalings='auto', title="Filtered data")

# Notch filter 16 Hz noise if necessary
Determine this with PSD visualization
For Pilot version 2: P001, P003, P007

In [ ]:
#Settings
notched = False

if notched:
    filt_raw_2 = filt_raw.copy().notch_filter(freqs=16, picks=ch_names)
    filt_raw_2.plot()

else:
    filt_raw_2 = filt_raw.copy()

# Automatically Detect Bad Channels
Using RANSAC bad channel detection built into `MNE` [link here](https://autoreject.github.io/stable/auto_examples/plot_ransac.html).
- Create RANSAC model
    * min_cor: Minimum correlation between channels to be considered as a good channel
    * n_resample: Number of resamples to use for RANSAC
    * min_channels: Minimum number of channels to be considered as a good channel
- Create 2 second epochs of whole timeseries EEG data
- Fit RANSAC model
    * Get the identified bad channels and remove them
    * Plot epoch averages before and after bad channel removal

In [ ]:
# Initialize RANSAC for bad channel detection
ransac = Ransac(verbose=True, picks="eeg", n_jobs=1, min_corr= 0.75, n_resample = 100, min_channels = 0.25)

# Create epochs with preloading enabled
epochs = mne.make_fixed_length_epochs(filt_raw_2, duration=2, overlap=0.5, preload=True)

# Fit RANSAC to detect bad channels
ransac.fit(epochs)

# Print the identified bad channels
bad_channels = ransac.bad_chs_
print("Bad channels detected by RANSAC:")
print('\n'.join(bad_channels))

# Plot the evoked data before removing bad channels
evoked_before = epochs.average()
fig_before = evoked_before.plot(ylim={"eeg": (-5000000, 5000000)})  # Set Y-axis limits for EEG channels
fig_before.suptitle("Evoked Data (Before Removing Bad Channels)", fontsize=16)

# Remove bad channels from the raw object
ica_raw_clean = filt_raw_2.copy().drop_channels(bad_channels)

# Create new epochs without bad channels
epochs_clean = mne.make_fixed_length_epochs(ica_raw_clean, duration=2, overlap=0.5, preload=True)

# Plot the evoked data after removing bad channels
evoked_after = epochs_clean.average() 
fig_after = evoked_after.plot(ylim={"eeg": (-5000000, 5000000)})
fig_after.suptitle("Evoked Data (After Removing Bad Channels)", fontsize=16)

# Save the channel list without bad channels
ch_names_clean = [ch for ch in ch_names if ch not in bad_channels]

# Epoch "On" Data

For P001 (pilot version 2), epoch end = "stimulus ended"
For all other data for pilot version 2, epoch end = "getting score"

In [ ]:
# Settings
list_of_events = []

for x in range(4):
    for y in range(3):
        list_of_events.append(f"Contrast{x+1}Size{y+1}")
epoch_end = "getting score"
#epoch_end = "stimulus ended"

[marker_ts, markers] = import_data.read_xdf_unity_markers(f"Data\\{file_name}")    # Import markers
#markers = markers[:-1] # Dan practice data has an extra marker at the end that is not needed
#marker_ts = marker_ts[:-1] # Remove the last marker timestamp and label

# Create Individual epochs for each event
[events_epochs, eeg_epochs] = data_tools.create_epochs(
    eeg_data = ica_raw_clean.get_data(), 
    eeg_ts = eeg_ts,
    markers = markers,
    markers_ts = marker_ts,
    events = list_of_events,
    epoch_end = epoch_end
    )

# Create a dict of stimuli using the unique events
dict_of_stimuli = {i: event for i, event in enumerate(list_of_events)}

# Organize epochs by stimuli and frequency
# Example: eeg_epochs_organized[stim_inx] returns all epochs for a specific stimulus.
        # eeg_epochs_organized[stim_inx].shape returns the shape of the epochs for a specific stimulus. like (2, 13, 2560) for 2 trials, 13 channels and 2560 samples (10 seconds).
eeg_epochs_organized = data_tools.epochs_stim(
    eeg_epochs = eeg_epochs,
    labels = events_epochs,
    stimuli = dict_of_stimuli
    )

for stim_idx, stim_label in dict_of_stimuli.items():
    stim_epochs = eeg_epochs_organized[stim_idx]
    
    if len(stim_epochs) > 0:
        print(f"Stimulus: {stim_label}, Shape: {stim_epochs.shape}")
    else:
        print(f"Stimulus: {stim_label}, No epochs found")

on_max_length = stim_epochs.shape[-1] # save the maximum length of the epochs to trim baseline epochs to this length

# Epoch "Off" Data

For pilot data 2 P001, after each "on" epoch, there are 2 markers. the first is "stimulus ended, getting score" and then the second is "stimulus ended".

Need to make an edit so there are not 80 "off" epochs found (2 for each off epoch where one goes from "stimulus ended, getting score"- "baseline ended" and the other goes from "stimulus ended"- "baseline ended" [this results in a bunch of overlap, need to get rid of epochs that start with "stimulus ended, getting score"])

In [ ]:
#for marker in markers:
#    if marker[0] == "stimulus ended, getting score":
#        marker[0] = "blah" 

In [ ]:
list_of_off_events = ["stimulus ended"] 
epoch_end = "baseline ended" 

# Create Individual epochs for each event
[baseline_epochs, eeg_baseline] = data_tools.create_epochs(
    eeg_data = ica_raw_clean.get_data(), 
    eeg_ts = eeg_ts,
    markers = markers,
    markers_ts = marker_ts,
    events = list_of_off_events,
    epoch_end = epoch_end,
    max_length = on_max_length,
    baseline = True
    )

dict_of_stimuli_2 = {0: "stimulus ended"}

baseline_eeg_epochs_organized = data_tools.epochs_stim(
    eeg_epochs = eeg_baseline,
    labels = baseline_epochs,
    stimuli = dict_of_stimuli_2
    )

for stim_idx, stim_label in dict_of_stimuli_2.items():
    stim_epochs = baseline_eeg_epochs_organized[stim_idx]
    
    if len(stim_epochs) > 0:
        print(f"Stimulus: {stim_label}, Shape: {stim_epochs.shape}")
    else:
        print(f"Stimulus: {stim_label}, No epochs found")

# Save Baseline data to an npz file
Can't use regular npy because each stimulus may have a different number of epochs (i.e. a different shape)

In [ ]:
save_data = True

if save_data:
    npz_file_name = file_name.split("\\")[-1].split(".")[0]

    # Create a dictionary to store each stimulus separately
    save_dict = {stim_label: np.array(baseline_eeg_epochs_organized[stim_idx]) 
                 for stim_idx, stim_label in dict_of_stimuli_2.items()}

    # Save using np.savez to preserve structure
    np.savez(f"Data\\PedsData\\EEG\\sub-P042\\ses-S001\\eeg\\{npz_file_name}_baseline.npz", **save_dict)

# Save Organized Epochs to an npz file

In [ ]:
save_data = True

if save_data:
    npz_file_name = file_name.split("\\")[-1].split(".")[0]

    # Create a dictionary to store each stimulus separately
    save_dict = {stim_label: np.array(eeg_epochs_organized[stim_idx]) 
                 for stim_idx, stim_label in dict_of_stimuli.items()}

    # Save using np.savez to preserve structure
    np.savez(f"Data\\PedsData\\EEG\\sub-P042\\ses-S001\\eeg\\{npz_file_name}.npz", **save_dict)

# Save settings to json

In [ ]:
# Settings
save_model = True 

if (save_model):
    json_data = {
        "file_name": file_name,
        "eeg_srate": eeg_fs,
        "ch_names": ch_names,
        "bad_chans": bad_channels,
        "new_ch_names": ch_names_clean,
        "notched": notched,
        "hpf_fc": hpf_fc,
        "lpf_fc": lpf_fc,s
        "labels": markers,
        "stimuli": dict_of_stimuli
        }

    # Get file name without extension
    json_file_name = file_name.split("\\")[-1].split(".")[0]
    
    # Write dictionary to json file
    with open(f"Data\\PedsData\\EEG\\sub-P042\\ses-S001\\eeg\\{json_file_name}.json", "w") as f:
        json.dump(json_data, f)